<a href="https://colab.research.google.com/github/Leungc2000/ai_cooker/blob/main/data_collection/all_recipe_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install recipe-scrapers

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.3/209.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.5/97.5 kB 5.0 MB/s eta 0:00:00
  Created wheel for jstyleson: filename=jstyleson-0.0.2-py3-none-any.whl size=2385 sha256=23a03030acb66fd533bdd79550a8abdc1e2ab712d21713f5eefa9a15ce359c09
  Stored in directory: /root/.cache/pip/wheels/12/51/c6/a1e751db88203e11c6d9ffe4683ca3d8c14b1479639bec1006
Successfully built jstyleson


In [ ]:
import requests
import re
import pandas as pd
from recipe_scrapers import scrape_html
from bs4 import BeautifulSoup

In [ ]:
# scrape all categories links
category_list = []

response = requests.get('https://www.allrecipes.com/recipes-a-z-6735880')
soup = BeautifulSoup(response.content, 'html.parser')
links = soup.find_all('a', attrs={'aria-label': lambda label: label and 'Read more about' in label})
category_list = [link['href'] for link in links]

In [ ]:
all_recipes_df = pd.DataFrame()
for category in category_list:
  # scrape recipe links on one page
  url_list = []
  response = requests.get(category)
  soup = BeautifulSoup(response.content, 'html.parser')
  links = soup.find_all('a', href=True, id=re.compile("mntl-card-list-items"))
  url_list = [link['href'] for link in links]

  # use scraper to get specific information from recipe link
  recipes = []

  for url in url_list:
    html = requests.get(url, headers={"User-Agent": "xxx"}).content
    scraper = scrape_html(html, org_url=url)
    cur_recipe = scraper.to_json()
    recipes.append(cur_recipe)

  all_recipes_df = pd.concat([all_recipes_df, pd.DataFrame(recipes)], ignore_index=True)

In [ ]:
all_recipes_df.to_csv('all_recipes.csv')

In [ ]:
help(scraper)

Help on AllRecipesCurated in module recipe_scrapers.allrecipes object:

class AllRecipesCurated(recipe_scrapers._abstract.AbstractScraper)
 |  AllRecipesCurated(html: str, url: str)
 |  
 |  Method resolution order:
 |      AllRecipesCurated
 |      recipe_scrapers._abstract.AbstractScraper
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(self, html: str, url: str)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  author(self)
 |      Author of the recipe.
 |  
 |  canonical_url(self)
 |      Canonical or original URL of the recipe.
 |  
 |  category(self)
 |      Category of the recipe.
 |  
 |  cook_time(self)
 |      Cooking time in minutes.
 |  
 |  cooking_method(self)
 |      The method of cooking the recipe
 |  
 |  cuisine(self)
 |      Cuisine of the recipe.
 |  
 |  description(self)
 |      Description of the recipe.
 |  
 |  dietary_restrictions(self)
 |      The specified dietary restrictions or guidelines for which thi